# PINN Vancomycin — Sensitivity Analysis (Kaggle, 2×T4)

Notebook za pokretanje glavnog eksperimenta na Kaggle GPU sesiji.

**Ključne razlike u odnosu na Colab verziju:**
- Nema Google Drive-a → rezultati idu u `/kaggle/working/` (automatski dostupni za download)
- 2×T4 GPU: `cuda:0` i `cuda:1` — 1-odeljni i 2-odeljni model rade **paralelno** (threading)
- Kaggle sesija traje do **12h** — dovoljno za oba modela paralelno

**Podešavanja pre pokretanja:**
1. `Settings → Accelerator → GPU T4 x2`
2. `Settings → Persistence → Variables and Files` (čuva `/kaggle/working/` između sesija)
3. U ćeliji 2: postavi `REPO_URL` na tvoj GitHub link

---
| Ćelija | Akcija | Trajanje |
|--------|--------|----------|
| 1 | GPU check + putanje | 5s |
| 2 | Kloniranje repoa + instalacija | 1–2 min |
| 3 | Generisanje podataka | 30s |
| 4 | Import modula | 5s |
| 5 | Test run (provjera) | ~5 min |
| 6 | **Paralelni eksperiment** (oba modela, oba GPU-a) | ~3–5h |
| 7 | Provjera napretka (pokrenuti u bilo kom trenutku) | 5s |
| 8 | Restore checkpoint (samo ako se sesija prekinula) | 10s |

## Ćelija 1 — GPU check i putanje

In [ ]:
import torch
from pathlib import Path

# Provjeri dostupnost GPU-a
n_gpu = torch.cuda.device_count()
print(f'PyTorch: {torch.__version__}')
print(f'CUDA dostupna: {torch.cuda.is_available()}')
print(f'Broj GPU-a: {n_gpu}')
for i in range(n_gpu):
    print(f'  cuda:{i} → {torch.cuda.get_device_name(i)}')

if n_gpu < 2:
    print()
    print('UPOZORENJE: manje od 2 GPU-a. Provjeri Settings → Accelerator → GPU T4 x2.')
    print('Kod ce raditi i sa 1 GPU (oba modela idu na cuda:0), ali sporije.')

# Putanje specifične za Kaggle
# /kaggle/working/ je izlazni folder — fajlovi su dostupni za download
OUT_DIR  = Path('/kaggle/working/pinn_results')
REPO_DIR = Path('/kaggle/working/pinn-vancomycin')
OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'\nIzlazni folder: {OUT_DIR}')
print(f'Repo folder:    {REPO_DIR}')

## Ćelija 2 — Kloniranje repoa i instalacija

⚠️ **Izmijeni `REPO_URL`** na tvoj GitHub link.

In [ ]:
import subprocess
from pathlib import Path

REPO_URL = 'https://github.com/YOUR_USERNAME/YOUR_REPO.git'   # <-- IZMIJENI OVO
REPO_DIR = Path('/kaggle/working/pinn-vancomycin')

if not REPO_DIR.exists():
    !git clone $REPO_URL {REPO_DIR}
else:
    print('Repo vec postoji, povlacim promjene...')
    !git -C {REPO_DIR} pull

!git -C {REPO_DIR} log --oneline -3

# scipy i seaborn nisu pre-instalirani na Kaggle, ostalo jeste
!pip install scipy seaborn -q

print('\nInstalacija gotova.')

## Ćelija 3 — Generisanje sintetičkih podataka

In [ ]:
!python {REPO_DIR}/src/data_processing.py

## Ćelija 4 — Import modula i konfiguracija

In [ ]:
import sys
import importlib.util
import torch
from pathlib import Path

REPO_DIR = Path('/kaggle/working/pinn-vancomycin')
OUT_DIR  = Path('/kaggle/working/pinn_results')
OUT_DIR.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(REPO_DIR / 'src'))

# Učitaj sensitivity modul (naziv počinje cifrom, pa ne može direktno import)
def _load(path, name):
    spec = importlib.util.spec_from_file_location(name, path)
    mod  = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod

sens = _load(REPO_DIR / 'experiments' / '03_sensitivity_analysis.py', 'sensitivity')

# Odabir device-a
n_gpu   = torch.cuda.device_count()
DEVICE0 = 'cuda:0' if n_gpu >= 1 else 'cpu'
DEVICE1 = 'cuda:1' if n_gpu >= 2 else DEVICE0   # fallback na isti GPU ako je samo 1

print(f'1-odeljni model → {DEVICE0}')
print(f'2-odeljni model → {DEVICE1}')
print(f'Checkpoint folder: {OUT_DIR}')
print('Moduli učitani.')

## Ćelija 5 — Test run (preporučeno pri prvom pokretanju)

Pokreće 8 kombinacija (N∈{3,5}, σ∈{0.0,0.05}, 2 seed-a) sa minimalnim brojem epoha.  
Trajanje: ~5 min. Ako prođe bez greške, sve je ispravno.

> Pokreće oba modela paralelno, svaki na svom GPU-u.

In [ ]:
import threading

TEST_DIR = OUT_DIR / 'test'
TEST_DIR.mkdir(parents=True, exist_ok=True)

errors = []

def _run_test_1comp():
    try:
        sens.run_sensitivity_1comp(test_mode=True, out_dir=TEST_DIR, device=DEVICE0)
    except Exception as e:
        errors.append(f'1comp: {e}')

def _run_test_2comp():
    try:
        sens.run_sensitivity_2comp(test_mode=True, out_dir=TEST_DIR, device=DEVICE1)
    except Exception as e:
        errors.append(f'2comp: {e}')

print('Pokretanje test run-a (paralelno na oba GPU-a)...')
t1 = threading.Thread(target=_run_test_1comp)
t2 = threading.Thread(target=_run_test_2comp)
t1.start(); t2.start()
t1.join();  t2.join()

if errors:
    print('\nGRESKE:')
    for e in errors:
        print(f'  {e}')
else:
    print('\nTest prosao bez gresaka. Brisem test CSV-ove...')
    import shutil
    shutil.rmtree(TEST_DIR, ignore_errors=True)
    print('Spreman za pravi eksperiment.')

## Ćelija 6 — Paralelni eksperiment (oba modela, oba GPU-a)

**1200 runova ukupno** (600 po modelu). Oba modela rade **istovremeno** na zasebnim GPU-ima.

Procijenjeno trajanje (2×T4):
- 1-odeljni model (cuda:0): ~1–2h
- 2-odeljni model (cuda:1): ~3–5h  
- **Ukupno (paralelno): ~3–5h** umjesto 4–7h sekvencijalno

Checkpoint: svaki run se odmah upisuje u `OUT_DIR`. Prekid ne gubi napredak —  
pri restartovanju, pokrenuti ćeliju 8 (Restore), pa ponovo ovu ćeliju.

In [ ]:
import threading

errors = []

def _run_1comp():
    try:
        print(f'[1-comp] Startuje na {DEVICE0}')
        sens.run_sensitivity_1comp(test_mode=False, out_dir=OUT_DIR, device=DEVICE0)
        print(f'[1-comp] ZAVRSENO')
    except Exception as e:
        errors.append(f'1comp: {e}')
        print(f'[1-comp] GRESKA: {e}')

def _run_2comp():
    try:
        print(f'[2-comp] Startuje na {DEVICE1}')
        sens.run_sensitivity_2comp(test_mode=False, out_dir=OUT_DIR, device=DEVICE1)
        print(f'[2-comp] ZAVRSENO')
    except Exception as e:
        errors.append(f'2comp: {e}')
        print(f'[2-comp] GRESKA: {e}')

t1 = threading.Thread(target=_run_1comp, name='1comp')
t2 = threading.Thread(target=_run_2comp, name='2comp')

t1.start()
t2.start()

# Blokira dokle god oba nisu gotova
t1.join()
t2.join()

print()
if errors:
    print('GRESKE:')
    for e in errors:
        print(f'  {e}')
else:
    print('Oba modela zavrsena.')
    print(f'Rezultati su u: {OUT_DIR}')
    print('Fajlovi su dostupni u Output tabu desno (pored Run All dugmeta).')

## Ćelija 7 — Provjera napretka

Pokrenuti u bilo kom trenutku tokom eksperimenta (u novoj ćeliji ili dok ćelija 6 radi).

In [ ]:
import pandas as pd
from pathlib import Path

OUT_DIR = Path('/kaggle/working/pinn_results')
TOTAL   = 5 * 4 * 30   # N x sigma x seed = 600 po modelu

for name in ['sensitivity_1comp.csv', 'sensitivity_2comp.csv']:
    p = OUT_DIR / name
    if not p.exists():
        print(f'{name}: ne postoji jos')
        continue
    df = pd.read_csv(p)
    print(f'{name}: {len(df)}/{TOTAL} ({len(df)/TOTAL*100:.1f}%)')
    if len(df) > 0:
        err_col = 'pinn_err_k10'
        ok = df[df['status'] == 'ok']
        print(f'  Uspjesnih: {len(ok)}/{len(df)}')
        if err_col in ok.columns and len(ok) > 0:
            print(f'  Median PINN err_k10:  {ok[err_col].median():.2f}%')
            print(f'  Median bench err_k10: {ok["bench_err_k10"].median():.2f}%')
        print(f'  Po N: {ok["N"].value_counts().sort_index().to_dict()}')
    print()

## Ćelija 8 — Restore checkpoint (samo ako se sesija prekinula)

**Kada koristiti:** sesija je istekla, ali si sačuvala CSV-ove (vidi dolje kako).  
Pokreni ćelije 1–4, pa ovu ćeliju, pa ponovo ćeliju 6 — preskočiće već završene kombinacije.

**Kako sačuvati CSV-ove između sesija:**  
Kaggle čuva `/kaggle/working/` samo dok je sesija aktivna. Opcije:
1. **Preuzmi ručno**: `Output` tab → klikni na CSV fajl → `Download`
2. **Kaggle Dataset**: napravi dataset iz output fajlova i priloži ga novoj sesiji
3. **Postavi `Persistence: Variables and Files`** u Settings (čuva `/kaggle/working/` između sesija iste verzije notebooka)

In [ ]:
import shutil
from pathlib import Path

OUT_DIR = Path('/kaggle/working/pinn_results')
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Ako si prethodno sačuvala CSV-ove kao Kaggle Dataset i priložila ih,
# postaviće se u /kaggle/input/<dataset-name>/
# Izmijeni putanju ispod prema stvarnom imenu dataseta:
INPUT_DIR = Path('/kaggle/input/pinn-checkpoint')   # <-- izmijeni naziv dataseta

if not INPUT_DIR.exists():
    print(f'Input dataset nije priložen: {INPUT_DIR}')
    print('Priloži dataset sa checkpoint CSV-ovima u notebook Settings → Add data.')
else:
    restored = 0
    for csv in sorted(INPUT_DIR.glob('sensitivity_*.csv')):
        dst = OUT_DIR / csv.name
        shutil.copy(csv, dst)
        rows = sum(1 for _ in open(dst)) - 1
        print(f'  Restored: {csv.name}  ({rows} redova)')
        restored += 1
    if restored == 0:
        print('Nema sensitivity_*.csv fajlova u priloženom datasetu.')
    else:
        print(f'\nRestore zavrsen. Nastavi sa celijom 6.')